# TECLA — Extracción terminológica basada en corpus

Este cuaderno muestra el flujo completo de TECLA:

1. Instalación y configuración
2. Subida de archivos (PDF o TXT)
3. Construcción y limpieza del corpus
4. Extracción de candidatos a término
5. Visualización y exportación de resultados
6. Opciones avanzadas

---
> **Requisito previo:** tener el repositorio TECLA clonado o subido a `/content/TECLA`.

## 1. Instalación

In [ ]:
# Clona el repositorio (solo la primera vez)
!git clone https://github.com/tu-usuario/TECLA /content/TECLA

In [ ]:
import sys
sys.path.insert(0, "/content/TECLA")

from tecla.setup_colab import setup_tecla

# Cambia languages según tu corpus
# Opciones: 'es', 'en', 'fr', 'de', 'it', 'pt'  —  o 'all'
setup_tecla(path="/content/TECLA", languages=["es", "en"])

## 2. Subida de archivos

Puedes subir **PDFs** o **TXTs** ya limpios. TECLA acepta ambos formatos.

In [ ]:
import os
from google.colab import files

# Carpeta de trabajo
CORPUS_DIR = "/content/mi_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

print("Selecciona tus archivos (PDF o TXT):")
uploaded = files.upload()

for nombre, contenido in uploaded.items():
    ruta = os.path.join(CORPUS_DIR, nombre)
    with open(ruta, "wb") as f:
        f.write(contenido)
    print(f"  ✓ Guardado: {nombre}")

print(f"\nArchivos en {CORPUS_DIR}:")
print(os.listdir(CORPUS_DIR))

## 3. Construcción y limpieza del corpus

TECLA convierte los archivos a texto limpio antes de procesar.
Activa solo los filtros que necesites para tu tipo de corpus.

In [ ]:
from pathlib import Path
from tecla.corpus.builder import build_corpus, build_corpus_from_txt

corpus_path = Path(CORPUS_DIR)
out_folder  = str(corpus_path / "_tecla_txt")

# --- Opciones de limpieza ---
# Activa las que sean relevantes para tu corpus
opciones_limpieza = {
    "rejoin_hyphens_flag":              True,   # une palabras partidas por guión (OCR)
    "remove_section_headers_flag":      True,   # elimina títulos de sección sueltos
    "remove_keyword_lines_flag":        True,   # elimina líneas "Keywords: ..."
    "remove_captions_flag":             True,   # elimina pies de figura/tabla
    "remove_urls_flag":                 True,   # elimina líneas de URL/DOI/email
    "remove_publication_metadata_flag": True,   # elimina copyright, fechas, ISSN
    "remove_journal_metadata_flag":     True,   # elimina vol./núm./pp. de revista
    "remove_inline_citations_flag":     True,   # elimina [1], [2,3] inline
    "remove_statistical_lines_flag":    False,  # desactiva si el corpus tiene tablas útiles
    "remove_bibliography_flag":         True,   # elimina la sección de referencias
    "structure":                        True,   # elimina números de página y líneas cortas
    "remove_stopwords_flag":            False,  # NO activar para extracción terminológica
    "lemmatize_flag":                   False,  # spaCy lematiza en el paso de extracción
}

# Detecta si hay PDFs o TXTs
archivos = list(corpus_path.iterdir())
hay_pdfs = any(f.suffix.lower() == ".pdf" for f in archivos)

if hay_pdfs:
    print("PDFs detectados — convirtiendo a texto…")
    build_corpus(
        input_folder=str(corpus_path),
        out_folder=out_folder,
        clean=True,
        clean_params=opciones_limpieza,
        compute_stats=False,
    )
else:
    print("TXTs detectados — aplicando limpieza…")
    build_corpus_from_txt(
        input_folder=str(corpus_path),
        out_folder=out_folder,
        clean=True,
        clean_params=opciones_limpieza,
        compute_stats=False,
    )

carpeta_limpia = str(Path(out_folder) / "clean")
textos_limpios = os.listdir(carpeta_limpia)
print(f"\n✓ {len(textos_limpios)} documento(s) listos en: {carpeta_limpia}")

### 3b. Revisión de calidad (opcional)

Comprueba si quedaron líneas con ruido de discurso genérico.

In [ ]:
from tecla.corpus.cleaning import flag_discourse_noise

IDIOMA = "es"  # cambia según tu corpus

for nombre in textos_limpios[:3]:   # revisa los primeros 3 documentos
    ruta = os.path.join(carpeta_limpia, nombre)
    with open(ruta, encoding="utf-8") as f:
        texto = f.read()
    sospechosas = flag_discourse_noise(texto, language=IDIOMA)
    if sospechosas:
        print(f"\n{nombre} — {len(sospechosas)} línea(s) sospechosas:")
        for l in sospechosas[:5]:
            print(f"  · {l}")
    else:
        print(f"{nombre} — sin ruido detectado")

## 4. Procesamiento lingüístico y extracción de términos

In [ ]:
from tecla.corpus.processor import corpus_to_documents

IDIOMA = "es"          # idioma del corpus
MODELO = "es_core_news_md"   # modelo spaCy a usar

print("Cargando modelo lingüístico y procesando corpus…")
documentos = corpus_to_documents(
    corpus_folder=carpeta_limpia,
    language=IDIOMA,
    spacy_model=MODELO,
)

total_tokens = sum(len(d["tokens"]) for d in documentos)
print(f"✓ {len(documentos)} documento(s) — {total_tokens:,} tokens procesados")

In [ ]:
from tecla.extractor.hybrid_extractor import HybridTermExtractor

extractor = HybridTermExtractor(
    language=IDIOMA,
    weights={"tfidf": 0.6, "cvalue": 0.4},  # ajusta según el corpus
)

resultados = extractor.extract(
    documents=documentos,
    min_freq=2,          # frecuencia mínima en el corpus
    min_doc_freq=1,      # mínimo de documentos en que aparece
    max_span_tokens=6,   # longitud máxima del candidato (en tokens)
    # term_length=(2, None),  # descomenta para solo términos multipalabra
    # top_k=200,              # descomenta para quedarte con los 200 mejores
    # percentile=80,          # descomenta para top 20%
)

print(f"\n✓ {len(resultados)} candidatos extraídos")

## 5. Visualización de resultados

In [ ]:
import pandas as pd

df = pd.DataFrame(resultados)[
    ["term", "lemma", "length", "frequency", "doc_frequency",
     "tf_idf", "c_value", "nc_value", "terminologicity", "variants", "patterns"]
].sort_values("terminologicity", ascending=False)

print(f"Top 20 candidatos a término ({IDIOMA.upper()}):")
df.head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de scores
axes[0].hist(df["terminologicity"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribución de terminologicidad")
axes[0].set_xlabel("Score")

# Distribución de longitudes
df["length"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color="steelblue", edgecolor="white"
)
axes[1].set_title("Longitud de candidatos (tokens)")
axes[1].set_xlabel("Nº de tokens")

# Frecuencia de patrones POS
from collections import Counter
patron_counts = Counter(
    p for row in df["patterns"] for p in (row if isinstance(row, list) else [])
)
pd.Series(patron_counts).sort_values(ascending=True).plot(
    kind="barh", ax=axes[2], color="steelblue", edgecolor="white"
)
axes[2].set_title("Patrones POS más frecuentes")

plt.tight_layout()
plt.show()

## 6. Exportación de resultados

In [ ]:
from pathlib import Path
from tecla.extractor.hybrid_extractor import HybridTermExtractor

ruta_csv = Path("/content/terminos_extraidos.csv")
ruta_tbx = Path("/content/terminos_extraidos.tbx")

# Extrae y guarda directamente
extractor.extract(
    documents=documentos,
    min_freq=2,
    output_path=ruta_csv,
    tbx_output_path=ruta_tbx,
)

print(f"✓ CSV guardado en: {ruta_csv}")
print(f"✓ TBX guardado en: {ruta_tbx}")

# Descarga automática
from google.colab import files
files.download(str(ruta_csv))
files.download(str(ruta_tbx))

## 7. Opciones avanzadas

### 7a. Pesos de scoring

El score de terminologicidad combina TF-IDF y NC-value. Puedes ajustar el equilibrio:

In [ ]:
# Predefinidos:
# {"tfidf": 0.8, "cvalue": 0.2}  →  prioriza frecuencia
# {"tfidf": 0.5, "cvalue": 0.5}  →  equilibrado
# {"tfidf": 0.2, "cvalue": 0.8}  →  prioriza complejidad (CTs anidados)
# "variance"                      →  pesos automáticos por varianza
# "entropy"                       →  pesos automáticos por entropía

extractor_avanzado = HybridTermExtractor(
    language=IDIOMA,
    weights="variance",   # TECLA calcula los pesos óptimos automáticamente
)

resultados_avanzados = extractor_avanzado.extract(
    documents=documentos,
    min_freq=2,
    term_length=(2, None),   # solo multipalabra
    top_k=100,
)

pd.DataFrame(resultados_avanzados)[["term", "terminologicity", "frequency", "patterns"]].head(10)

### 7b. Gramática POS personalizada

Puedes modificar `tecla/extractor/grammar.yaml` para añadir o quitar patrones,
o pasar una gramática alternativa:

In [ ]:
from pathlib import Path

# Ruta a la gramática por defecto
grammar_path = Path("/content/TECLA/tecla/extractor/grammar.yaml")

# Muestra los patrones definidos para el idioma
import yaml
with open(grammar_path, encoding="utf-8") as f:
    grammar = yaml.safe_load(f)

print(f"Patrones definidos para '{IDIOMA}':")
for p in grammar.get(IDIOMA, []):
    print(f"  {p['id']:30s}  {p['pos_pattern']}")

### 7c. Stop-terms personalizadas

Puedes añadir términos genéricos de tu dominio que no quieras en la lista final:

In [ ]:
from tecla.extractor.stop_terms import is_stop_term

# Comprueba si un término está en la stoplist
for termino in ["et al", "análisis cualitativo", "extracción terminológica"]:
    resultado = is_stop_term(termino, language=IDIOMA)
    print(f"  '{termino}' → {'STOP' if resultado else 'candidato válido'}")

# Para filtrar con stop-terms extra en tiempo de extracción,
# puedes pasar `extra` a is_stop_term o añadir entradas a stop_es.txt

### 7d. Pipeline completo en una sola llamada

In [ ]:
from pathlib import Path
from pipeline.pipeline import run_tecla_pipeline

resultados_pipeline = run_tecla_pipeline(
    corpus_path=Path(CORPUS_DIR),
    language="es",
    cleaning_options={
        "rejoin_hyphens_flag":              True,
        "remove_section_headers_flag":      True,
        "remove_keyword_lines_flag":        True,
        "remove_captions_flag":             True,
        "remove_urls_flag":                 True,
        "remove_publication_metadata_flag": True,
        "remove_journal_metadata_flag":     True,
        "remove_inline_citations_flag":     True,
        "remove_bibliography_flag":         True,
        "structure":                        True,
    },
    extraction_options={
        "weights":          {"tfidf": 0.6, "cvalue": 0.4},
        "multiword_only":   False,
        "max_term_length":  6,
        "percentile":       None,
    },
)

print(f"\n✓ {len(resultados_pipeline)} términos extraídos")
pd.DataFrame(resultados_pipeline)[["term", "terminologicity", "frequency"]].head(10)